# Lesson 6 — How good is it? Cross-entropy & perplexity

A good language model gives high probability to real text *it hasn't seen*. Cross-entropy is the average
surprise per word, −(1/N) Σ log P(wᵢ | context). Perplexity is its exponential: roughly "how many equally likely
words is the model choosing between at each step". Lower is better, and it only counts on held-out text.

In [1]:
import re, math, random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

## A held-out split

I shuffle the 60 sentences with seed 0 and keep 80% (48) for training and 20% (12) for testing. The vocabulary
comes from training only, plus `</s>` and `<unk>`. `<unk>` just reserves one extra slot in |V| for unknown words;
test words aren't actually replaced, they simply get the smoothed count-0 probability.

In [4]:
sents = to_sentences(open("corpus.txt").read())
rng = random.Random(0); rng.shuffle(sents)
cut = int(0.8 * len(sents)); train, test = sents[:cut], sents[cut:]
V = set(t for s in train for t in to_tokens(s)) | {"</s>", "<unk>"}

## Scoring a dataset

For every word in every sentence: look up its context, get the add-k probability (never zero, so the log is
safe), and add up the log-probabilities. Perplexity is exp of the negative average.

In [5]:
def perplexity(foll, n, data, k=1.0):
    log_sum, N = 0.0, 0
    for s in data:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            ctx, w = tuple(toks[i:i + n - 1]), toks[i + n - 1]
            c = foll.get(ctx, Counter()); total = sum(c.values())
            p = (c[w] + k) / (total + k * len(V))              # add-k, so never zero
            log_sum += math.log(p); N += 1
    return math.exp(-log_sum / N)

In [6]:
bi, tri = build(train, 2), build(train, 3)
print(f"bigram  perplexity  ->  train {perplexity(bi, 2, train):6.1f}   held-out {perplexity(bi, 2, test):6.1f}")
print(f"trigram perplexity  ->  train {perplexity(tri, 3, train):6.1f}   held-out {perplexity(tri, 3, test):6.1f}")

bigram  perplexity  ->  train   52.7   held-out   94.1
trigram perplexity  ->  train   60.6   held-out  115.8


Two lessons here. First, train perplexity is much lower than held-out, because the model has partly memorised
its training sentences, so only the held-out number is honest. Second, the trigram is *worse* on held-out text
(115.8 vs 94.1) even though its generated text looked better in Lesson 4. With 48 training sentences most
two-word contexts are new, so it leans on smoothing and gets surprised. More context only pays off with more data.

## Test yourself

**1. What does a perplexity of ~94 mean intuitively?**
On average the model is as unsure as if it were choosing uniformly among about 94 words at each step. Lower means
it predicts the real next word with more confidence.

**2. Why must perplexity be measured on held-out text?**
On training text the model has already seen every n-gram, so the score rewards memorisation. Held-out text tests
whether it generalises to new sentences, which is the job it actually has.

**3. Why is the trigram's held-out perplexity worse than the bigram's here?**
Data sparsity. Most trigram contexts in the test set were never seen in 48 training sentences, so the probability
falls back to uniform smoothing (exactly 1/|V| for an unseen context). The bigram has seen most single-word contexts.

## My notes

The key idea: perplexity = exp(average negative log-probability) on unseen text, and lower is better. What surprised
me is that the trigram, which wrote nicer sentences, actually scores worse. Fluent output and a good model are not
the same thing.